# Medium Q3 — 2022 巴基斯坦 Indus Valley 洪水範圍分析

## 資料集選擇與說明

選用 **Sentinel-1 SAR GRD（`COPERNICUS/S1_GRD`）**：

| 屬性 | 說明 |
|------|------|
| 感測器類型 | C-band SAR（5.4 GHz）|
| 空間解析度 | 10 m |
| 重訪週期 | 6–12 天 |
| 特別優勢 | **不受雲層影響**，2022 年巴基斯坦洪災期間大量降雨/雲覆，光學影像幾乎無效 |
| 波段 | VV、VH（垂直極化）|

## 處理方法
- 使用 **VV 極化**：洪水區水面對 SAR 後向散射低 → VV < 閾值 = 淹水
- 閾值設定：−15 dB（常用水體偵測閾值）
- 為避免誤判：
  1. 疊加 **SRTM 坡度圖**，排除坡度 > 5° 的山坡（避免山體陰影誤判）
  2. 使用洪水前影像計算「永久水體」遮罩（JRC Global Surface Water），只保留新增洪水像素
  3. 採用形態學 opening 濾波去除雜訊孤立像素

In [ ]:
import ee
import geemap
import pandas as pd
import matplotlib.pyplot as plt

ee.Authenticate()
ee.Initialize(project='0')

In [2]:
# Indus Valley AOI（巴基斯坦信德省-俾路支斯坦交界）
aoi = ee.Geometry.Rectangle([66.0, 25.0, 72.0, 30.0])

# 時間設定
PRE_START  = '2022-05-01'
PRE_END    = '2022-06-30'
FLOOD_START = '2022-08-01'
FLOOD_END   = '2022-09-30'

def get_s1_vv(start, end, orbit='DESCENDING'):
    return (ee.ImageCollection('COPERNICUS/S1_GRD')
            .filterBounds(aoi)
            .filterDate(start, end)
            .filter(ee.Filter.eq('instrumentMode', 'IW'))
            .filter(ee.Filter.eq('orbitProperties_pass', orbit))
            .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
            .select('VV')
            .mean()
            .clip(aoi))

s1_pre   = get_s1_vv(PRE_START, PRE_END)
s1_flood = get_s1_vv(FLOOD_START, FLOOD_END)

print('Sentinel-1 SAR 影像建立完成')

Sentinel-1 SAR 影像建立完成


In [3]:
# 平滑處理（Focal Mean，減少 speckle 雜訊）
def smooth(image, radius=50):
    return image.focal_mean(radius, 'circle', 'meters')

s1_pre_sm   = smooth(s1_pre)
s1_flood_sm = smooth(s1_flood)

# 洪水偵測：VV < -15 dB 判定為水體
THRESHOLD = -15
flood_raw = s1_flood_sm.lt(THRESHOLD).rename('flood_raw')

In [4]:
# ── 避免誤判的三重過濾 ────────────────────────────────────────────────

# 1. 坡度過濾（排除坡度 > 5° 的地形）
dem = ee.Image('USGS/SRTMGL1_003')
slope = ee.Terrain.slope(dem)
flat_mask = slope.lt(5)

# 2. 永久水體遮罩（JRC Global Surface Water，排除本來就是水的區域）
jrc = ee.Image('JRC/GSW1_4/GlobalSurfaceWater').select('seasonality')
perm_water = jrc.gte(10)  # 10+ 個月出現水體 → 永久水體

# 3. 洪水前基線：前期同一條件下不是水體
pre_water = s1_pre_sm.lt(THRESHOLD)
new_flood_only = flood_raw.And(pre_water.Not())  # 新增淹水（非洪前水體）

# 綜合遮罩
flood_filtered = new_flood_only.And(flat_mask).And(perm_water.Not()).rename('flood')

# 形態學 opening（去除孤立雜訊）
flood_clean = (flood_filtered
               .focal_min(100, 'circle', 'meters')
               .focal_max(100, 'circle', 'meters')
               .rename('flood_clean'))

print('洪水偵測完成')

洪水偵測完成


In [5]:
# 估算淹水面積（km²）
flood_area_m2 = flood_clean.multiply(ee.Image.pixelArea()).reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=aoi,
    scale=10,
    maxPixels=1e12
).get('flood_clean').getInfo()

area_km2 = flood_area_m2 / 1e6
print(f'估算淹水面積: {area_km2:,.1f} km²')

EEException: Computation timed out.

In [ ]:
# 地圖視覺化
Map = geemap.Map(center=[27.5, 68.0], zoom=7)

# SAR 影像
sar_vis = {'min': -25, 'max': 0, 'palette': ['black', 'white']}
Map.addLayer(s1_pre_sm,   sar_vis, 'SAR VV 洪前（5-6月）')
Map.addLayer(s1_flood_sm, sar_vis, 'SAR VV 洪水期（8-9月）', shown=False)

# 淹水範圍（紅色）
Map.addLayer(
    flood_clean.updateMask(flood_clean),
    {'palette': ['red']},
    f'淹水範圍（估算 {area_km2:,.0f} km²）'
)

Map

In [ ]:
# 火前/火後 SAR 差異視覺化（dB 差異）
diff = s1_flood_sm.subtract(s1_pre_sm).rename('VV_diff_dB')

Map2 = geemap.Map(center=[27.5, 68.0], zoom=7)
Map2.addLayer(diff, {'min': -10, 'max': 5,
                     'palette': ['#8B0000','red','orange','white','lightblue','blue']},
              '洪水前後 VV 差異 (dB)  負值=後向散射降低=淹水')
Map2.addLayer(flood_clean.updateMask(flood_clean),
              {'palette': 'cyan'}, '淹水遮罩')
Map2

In [ ]:
print('===== 分析摘要 =====')
print(f'資料集     : Sentinel-1 SAR GRD（COPERNICUS/S1_GRD）')
print(f'波段       : VV，IW 模式，下降軌道')
print(f'水體閾值   : VV < {THRESHOLD} dB')
print(f'洪前基準期 : {PRE_START} – {PRE_END}')
print(f'洪水期     : {FLOOD_START} – {FLOOD_END}')
print(f'估算淹水面積: {area_km2:,.1f} km²')
print()
print('誤判抑制方法：')
print('  1. SRTM 坡度 > 5° 排除（避免山體陰影）')
print('  2. JRC 永久水體排除（避免將河湖誤計為洪水）')
print('  3. 洪前基線排除（僅保留新增水體）')
print('  4. 形態學 opening 濾波（去除孤立雜訊像素）')